# WAFER-GUARD · Fab-7 Excursion Detector — Repaired Pipeline

This notebook replaces the original release candidate. It fixes three silent defects:

| # | Defect | Where (original) | Fix here |
|---|---|---|---|
| 1 | **Data leakage**: imputer/scaler fit on all rows, SMOTE before the split, random split, balanced test set | §2 | Time-ordered splits; every statistic uses past rows only; no synthetic samples; test kept at natural prevalence |
| 2 | **Corrupted loss gradients**: `reconstructed` is `[B, F]` but `batch_x` is `[B, 1, F]`, so the MSE broadcasts to `[B, B, F]` | §3 + §5 | Flat `[B, F]` tensors; masked per-element MSE with explicit shape asserts |
| 3 | **Invariant blindness**: legacy probe station PS-L writes `sensor_0 = 0.0` (sentinel) and firmware codes 100/200/300 in `sensor_1`; the original treats both as measurements | §2 | Detect PS-L rows, one-hot encode the codes, mask both channels out of scaling, imputation and the reconstruction loss |

**Causality rules applied throughout**

* A wafer is scaled only with statistics from wafers that came **before** it (expanding window).
* A wafer in a validation or test window is scaled with statistics **frozen at the start of that window**, so the evaluation population never influences its own preprocessing.
* Training wafers from the **24 h** before an evaluation window are left out (label embargo): their electrical-test labels would not be known yet at prediction time.
* Settings, training length and decision threshold come from **walk-forward validation** on the development period. The test window is scored once at the end.

Section 10 reproduces each original defect with numbers from this run.

## 0 · Environment

In [1]:
# Install with:  pip install -r requirements.txt
import os, json, random, subprocess, sys, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)            # single thread -> identical numbers on any CPU
DEVICE = torch.device("cpu")

DATA_PATH   = "secom_telemetry_modified.csv"
TEST_FRAC   = 0.20                  # last 20 % of lots in time = held-out test window
BURN_IN     = 30                    # first lots only seed the expanding statistics
MIN_HISTORY = 30                    # a statistic needs >= 30 past observations to be used
EMBARGO     = np.timedelta64(24, "h")
N_FOLDS, FOLD_START = 5, 453        # walk-forward validation blocks inside the dev period
CLIP_Z      = 10.0
MAX_EPOCHS  = 40
BASE_HP = dict(batch_size=64, weight_decay=1e-2, dropout=0.3, lambda_clf=1.0)
GRID = [dict(hidden=h, latent=l, lr=lr) for h in (64, 256) for l in (8, 32) for lr in (1e-4, 1e-3)]
print(f"torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__} | device={DEVICE}")

torch 2.14.1+cu130 | numpy 2.5.3 | pandas 3.0.5 | device=cpu


## 1 · Data — loaded in time order

In [2]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Put {DATA_PATH} next to this notebook (it ships in the repository root).")
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
TS = df["timestamp"].to_numpy()

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_all = df[SENSOR_COLS].to_numpy(dtype=np.float64)             # [N, 590]
y_all = (df["label"].to_numpy() == 1).astype(np.int64)          # 1 = defective (label +1)
N = len(df)
assert set(np.unique(df["label"])) == {-1, 1}
print(f"X: {X_all.shape} | defect rate: {y_all.mean():.2%} ({y_all.sum()} fail / {N - y_all.sum()} pass)")
print(f"time range: {df.timestamp.min()} -> {df.timestamp.max()}")

X: (1567, 590) | defect rate: 6.64% (104 fail / 1463 pass)
time range: 2008-07-19 11:55:00 -> 2008-10-17 06:07:00


## 2 · Legacy probe station PS-L (tool-matching invariant)

PS-L firmware writes `sensor_0 == 0.0` exactly. That is a sentinel, not a measurement.
On those wafers `sensor_1` holds a firmware error code (100 / 200 / 300) instead of a
reading. The check is a fixed rule applied to one row at a time.

* `is_legacy` and a one-hot of the code become **categorical** inputs. They are never scaled.
* On PS-L rows, `sensor_0` and `sensor_1` are set to *missing*. They are left out of every
  statistic, are never imputed into the reconstruction target, and are masked out of the loss.

In [3]:
LEGACY_CODES = [100.0, 200.0, 300.0]
is_legacy = (X_all[:, 0] == 0.0)
codes = X_all[is_legacy, 1]
print(f"legacy wafers: {is_legacy.sum()} ({is_legacy.mean():.2%})")
print("sensor_1 values on legacy rows:", {float(k): int(v) for k, v in zip(*np.unique(codes, return_counts=True))})
assert np.isin(codes, LEGACY_CODES).all(), "unexpected PS-L firmware code"
assert not np.isin(X_all[~is_legacy, 1], LEGACY_CODES).any(), "firmware code on a non-legacy row"
print(f"non-legacy sensor_1 range: {np.nanmin(X_all[~is_legacy,1]):.1f} .. {np.nanmax(X_all[~is_legacy,1]):.1f}")
print(f"defect rate  legacy={y_all[is_legacy].mean():.2%}  current-gen={y_all[~is_legacy].mean():.2%}")

CAT = np.zeros((N, 1 + len(LEGACY_CODES)), dtype=np.float32)
CAT[:, 0] = is_legacy
for j, c in enumerate(LEGACY_CODES):
    CAT[:, 1 + j] = is_legacy & (X_all[:, 1] == c)
CAT_NAMES = ["is_legacy"] + [f"ps_l_code_{int(c)}" for c in LEGACY_CODES]

X_cont = X_all.copy()
X_cont[is_legacy, 0] = np.nan       # sentinel, not a measurement
X_cont[is_legacy, 1] = np.nan       # firmware code, not a measurement

legacy wafers: 146 (9.32%)
sensor_1 values on legacy rows: {100.0: 61, 200.0: 35, 300.0: 50}
non-legacy sensor_1 range: 2158.8 .. 2846.4
defect rate  legacy=0.00%  current-gen=7.32%


## 3 · Time windows: burn-in, walk-forward folds, embargo, test

```
| burn-in | ........ development period (train + walk-forward validation) ........ | test (last 20 %) |
           train ->  [ val 1 ][ val 2 ][ val 3 ][ val 4 ][ val 5 ]
```

Fold *k* trains on every development lot before validation block *k*, minus the 24 h embargo,
and is scored on block *k*. The final model trains on the whole development period, minus the
24 h before the test window.

In [4]:
def no_tie(i):
    """Move a boundary forward so two lots with the same timestamp never sit on both sides."""
    while 0 < i < N and TS[i] == TS[i - 1]:
        i += 1
    return i

def train_end(eval_start):
    """Index of the first lot inside the label embargo before an evaluation window."""
    return int(np.searchsorted(TS, TS[eval_start] - EMBARGO, side="left"))

N_DEV = no_tie(int(round(N * (1 - TEST_FRAC))))
blk = (N_DEV - FOLD_START) // N_FOLDS
FOLDS = []
for k in range(N_FOLDS):
    s = no_tie(FOLD_START + k * blk)
    e = N_DEV if k == N_FOLDS - 1 else no_tie(FOLD_START + (k + 1) * blk)
    FOLDS.append(dict(train=np.arange(BURN_IN, train_end(s)), val=np.arange(s, e), freeze=s))
TEST = dict(train=np.arange(BURN_IN, train_end(N_DEV)), val=np.arange(N_DEV, N), freeze=N_DEV)

def describe(name, f):
    tr, va = f["train"], f["val"]
    print(f"{name:6s} train {tr[0]:4d}-{tr[-1]:4d} (n={len(tr):4d}, defects={y_all[tr].sum():3d}) | "
          f"eval {va[0]:4d}-{va[-1]:4d} (n={len(va):3d}, defects={y_all[va].sum():2d}, {y_all[va].mean():.1%}) | "
          f"embargoed lots={va[0] - tr[-1] - 1}")
    assert TS[tr[-1]] < TS[va[0]] - EMBARGO
for k, f in enumerate(FOLDS):
    describe(f"fold {k+1}", f)
describe("TEST", TEST)
print(f"pooled validation defects: {sum(y_all[f['val']].sum() for f in FOLDS)}")

fold 1 train   30- 449 (n= 420, defects= 53) | eval  453- 612 (n=160, defects= 7, 4.4%) | embargoed lots=3
fold 2 train   30- 597 (n= 568, defects= 58) | eval  613- 772 (n=160, defects= 2, 1.2%) | embargoed lots=15
fold 3 train   30- 771 (n= 742, defects= 62) | eval  773- 932 (n=160, defects= 9, 5.6%) | embargoed lots=1
fold 4 train   30- 894 (n= 865, defects= 67) | eval  933-1092 (n=160, defects= 2, 1.2%) | embargoed lots=38
fold 5 train   30-1068 (n=1039, defects= 73) | eval 1093-1253 (n=161, defects= 9, 5.6%) | embargoed lots=24
TEST   train   30-1221 (n=1192, defects= 78) | eval 1254-1566 (n=313, defects=17, 5.4%) | embargoed lots=32
pooled validation defects: 29


## 4 · Causal preprocessing — expanding-window robust scaling

* **Column set**: decided from the burn-in lots only. A column is kept if it is ≥ 50 % observed
  and not constant there.
* **Statistics**: for lot *t*, the median, IQR and standard deviation come from lots `0 … t−1` only.
  Inside an evaluation window, the statistics stay frozen at the window start.
* **Scaling**: `z = (x − median) / (IQR / 1.349)`, falling back to the standard deviation when
  IQR = 0. Median and IQR are robust, so a few extreme PS-L outliers cannot distort the scale
  for everyone else. `z` is clipped to ±10.
* **Missing values**: a cell is *usable* only if it was observed and its statistic has ≥ 30 past
  observations. Unusable cells are set to 0, and the 0/1 **mask is given to the encoder** as an
  input, so the model can tell "missing" from "exactly average". The mask also removes those
  cells from the reconstruction loss.

In [5]:
burn = X_cont[:BURN_IN]
frac_obs = np.mean(~np.isnan(burn), axis=0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    burn_std = np.nanstd(burn, axis=0)
KEEP = (frac_obs >= 0.5) & (np.nan_to_num(burn_std, nan=0.0) > 0)
KEPT_NAMES = [n for n, k in zip(SENSOR_COLS, KEEP) if k]
XK = X_cont[:, KEEP]
F_CONT = XK.shape[1]
print(f"continuous channels kept: {F_CONT} / {len(SENSOR_COLS)} "
      f"(sensor_0 kept={KEEP[0]}, sensor_1 kept={KEEP[1]})")

# row t of each table = statistic over lots 0..t-1   (shift(1) => strictly earlier lots)
_exp = pd.DataFrame(XK).expanding(min_periods=1)
MED = _exp.quantile(0.50).shift(1).to_numpy()
IQR = (_exp.quantile(0.75) - _exp.quantile(0.25)).shift(1).to_numpy()
SD  = _exp.std().shift(1).to_numpy()
CNT = _exp.count().shift(1).fillna(0).to_numpy()

def stats_at(t):
    """Scaling statistics built only from lots before index t (t may be an array)."""
    t = np.asarray(t)
    med, iqr, sd, cnt = MED[t], IQR[t], SD[t], CNT[t]
    scale = np.where(np.nan_to_num(iqr) / 1.349 > 1e-12, iqr / 1.349, sd)
    usable = (cnt >= MIN_HISTORY) & (np.nan_to_num(scale) > 1e-12)
    return med, np.where(usable, scale, 1.0), usable

def transform(idx, freeze_at):
    """Scale lots `idx`; any lot at or after `freeze_at` uses statistics frozen at `freeze_at`."""
    idx = np.asarray(idx)
    med, scale, usable = stats_at(np.minimum(idx, freeze_at))
    x = XK[idx]
    mask = ~np.isnan(x) & usable
    with np.errstate(invalid="ignore"):
        z = np.where(mask, (x - med) / scale, 0.0)
    z = np.clip(z, -CLIP_Z, CLIP_Z)
    return z.astype(np.float32), mask.astype(np.float32)

def build(idx, freeze_at):
    z, m = transform(idx, freeze_at)
    return z, m, CAT[idx], y_all[idx].astype(np.float32)

# causality check: statistics at a window start must be identical when every later lot is deleted
for _s in [f["freeze"] for f in FOLDS] + [TEST["freeze"]]:
    _cut = pd.DataFrame(XK[:_s]).expanding(min_periods=1)
    assert np.allclose(_cut.quantile(0.50).to_numpy()[-1], MED[_s], equal_nan=True)
    assert np.allclose(_cut.std().to_numpy()[-1], SD[_s], equal_nan=True)
print("causality check passed: statistics never use lots at or after the point they are applied")
z_tr, m_tr = transform(TEST["train"], TEST["freeze"])
print(f"usable cells in final training set: {m_tr.mean():.2%}; "
      f"clipped cells: {(np.abs(z_tr) >= CLIP_Z).mean():.3%}")

continuous channels kept: 443 / 590 (sensor_0 kept=True, sensor_1 kept=True)


causality check passed: statistics never use lots at or after the point they are applied
usable cells in final training set: 97.42%; clipped cells: 0.462%


## 5 · Model — supervised autoencoder

The encoder sees `[scaled values ‖ usable-mask ‖ PS-L categorical]`. The decoder rebuilds only
the scaled values, and the reconstruction loss counts only usable cells.

In [6]:
class WaferSAE(nn.Module):
    def __init__(self, d_cont, d_cat, d_hid, d_lat, p_drop):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(2 * d_cont + d_cat, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_cont),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, z, mask, cat):
        assert z.ndim == 2 and z.shape == mask.shape and cat.ndim == 2, (z.shape, mask.shape, cat.shape)
        h = self.encoder(torch.cat([z, mask, cat], dim=1))
        return self.decoder(h), self.head(h).squeeze(-1), h        # [B,F], [B], [B,lat]


def masked_mse(recon, target, mask):
    assert recon.shape == target.shape == mask.shape, (recon.shape, target.shape, mask.shape)
    return ((recon - target) ** 2 * mask).sum() / mask.sum().clamp_min(1.0)


@torch.no_grad()
def predict(model, z, m, c):
    model.eval()
    _, logits, _ = model(torch.from_numpy(z), torch.from_numpy(m), torch.from_numpy(c))
    return torch.sigmoid(logits).numpy()


def fit_sae(train_idx, freeze_at, hp, epochs, seed=SEED, eval_idx=None):
    """Train on `train_idx`. If `eval_idx` is given, return its predictions after every epoch."""
    seed_everything(seed)
    z, m, c, y = build(train_idx, freeze_at)
    if eval_idx is not None:
        ze, me, ce, _ = build(eval_idx, freeze_at)
    model = WaferSAE(F_CONT, CAT.shape[1], hp["hidden"], hp["latent"], BASE_HP["dropout"]).to(DEVICE)
    pos_weight = torch.tensor((1 - y).sum() / max(y.sum(), 1), dtype=torch.float32)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=BASE_HP["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)   # same schedule for every run length
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(*map(torch.from_numpy, (z, m, c, y))),
                        batch_size=BASE_HP["batch_size"], shuffle=True, generator=g)
    per_epoch, history = [], []
    for _ in range(epochs):
        model.train(); tot = 0.0
        for zb, mb, cb, yb in loader:
            recon, logits, _ = model(zb, mb, cb)
            assert logits.shape == yb.shape, (logits.shape, yb.shape)
            loss = masked_mse(recon, zb, mb) + BASE_HP["lambda_clf"] * bce(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step(); tot += loss.item() * len(yb)
        sched.step(); history.append(tot / len(y))
        if eval_idx is not None:
            per_epoch.append(predict(model, ze, me, ce))
    return model, per_epoch, history

print(f"input width = {2 * F_CONT + CAT.shape[1]} (scaled {F_CONT} + mask {F_CONT} + categorical {CAT.shape[1]})")

input width = 890 (scaled 443 + mask 443 + categorical 4)


## 6 · Walk-forward model selection

For each of the 8 settings in `GRID`, each fold trains for 40 epochs and saves its validation
predictions after every epoch. The pooled validation predictions from all 5 folds (29 defects)
are scored with PR-AUC. The best (setting, epoch) pair wins.

Pooling gives far more defects than the single 11-defect window used before.

In [7]:
y_oof = np.concatenate([y_all[f["val"]] for f in FOLDS])
search = []
for hp in GRID:
    fold_preds = [fit_sae(f["train"], f["freeze"], hp, MAX_EPOCHS, eval_idx=f["val"])[1] for f in FOLDS]
    for ep in range(MAX_EPOCHS):
        oof = np.concatenate([fp[ep] for fp in fold_preds])
        search.append(dict(**hp, epoch=ep + 1, oof_ap=average_precision_score(y_oof, oof),
                           _oof=oof))
res = pd.DataFrame(search)
best_per_hp = res.loc[res.groupby(["hidden", "latent", "lr"])["oof_ap"].idxmax()]
print(best_per_hp.drop(columns="_oof").sort_values("oof_ap", ascending=False).to_string(index=False))
BEST = res.loc[res["oof_ap"].idxmax()]
HP = dict(hidden=int(BEST.hidden), latent=int(BEST.latent), lr=float(BEST.lr))
EPOCHS = int(BEST.epoch)
print(f"\nselected: {HP} | epochs={EPOCHS} | pooled validation PR-AUC={BEST.oof_ap:.4f} "
      f"(no-skill = {y_oof.mean():.4f})")

 hidden  latent     lr  epoch   oof_ap
    256      32 0.0010     20 0.090120
     64       8 0.0010     25 0.078972
    256       8 0.0010     20 0.062530
     64      32 0.0010     21 0.051690
     64      32 0.0001      4 0.043100
    256      32 0.0001     39 0.041238
    256       8 0.0001     34 0.041005
     64       8 0.0001     20 0.033526

selected: {'hidden': 256, 'latent': 32, 'lr': 0.001} | epochs=20 | pooled validation PR-AUC=0.0901 (no-skill = 0.0362)


## 7 · Decision threshold — chosen on pooled validation predictions

The threshold that maximises F1 on the pooled out-of-fold predictions of the selected
(setting, epoch) is frozen here, before the test window is scored.

In [8]:
OOF = BEST["_oof"]
prec_c, rec_c, thr_c = precision_recall_curve(y_oof, OOF)
f1_c = 2 * prec_c[:-1] * rec_c[:-1] / np.clip(prec_c[:-1] + rec_c[:-1], 1e-12, None)
THRESH = float(thr_c[np.nanargmax(f1_c)])
oof_pred = (OOF >= THRESH).astype(int)
print(f"frozen threshold = {THRESH:.4f}")
print(f"validation at that threshold: F1={f1_score(y_oof, oof_pred):.4f} "
      f"precision={precision_score(y_oof, oof_pred, zero_division=0):.4f} "
      f"recall={recall_score(y_oof, oof_pred):.4f} alarm rate={oof_pred.mean():.2%}")
for k, f in enumerate(FOLDS):
    lo = sum(len(FOLDS[j]["val"]) for j in range(k)); hi = lo + len(f["val"])
    yk = y_oof[lo:hi]
    ap = average_precision_score(yk, OOF[lo:hi]) if yk.sum() else float("nan")
    print(f"  fold {k+1}: defects={yk.sum():2d}  PR-AUC={ap:.3f}  (no-skill {yk.mean():.3f})")

frozen threshold = 0.0033
validation at that threshold: F1=0.1202 precision=0.0686 recall=0.4828 alarm rate=25.47%
  fold 1: defects= 7  PR-AUC=0.204  (no-skill 0.044)
  fold 2: defects= 2  PR-AUC=0.066  (no-skill 0.013)
  fold 3: defects= 9  PR-AUC=0.124  (no-skill 0.056)
  fold 4: defects= 2  PR-AUC=0.092  (no-skill 0.013)
  fold 5: defects= 9  PR-AUC=0.052  (no-skill 0.056)


## 8 · Final model — refit on the whole development period, scored once on the test window

In [9]:
final_model, _, final_hist = fit_sae(TEST["train"], TEST["freeze"], HP, EPOCHS)
print(f"trained {EPOCHS} epochs on {len(TEST['train'])} lots | final train loss {final_hist[-1]:.4f}")

z_te, m_te, c_te, y_te = build(TEST["val"], TEST["freeze"])
y_te = y_te.astype(int)
p_te = predict(final_model, z_te, m_te, c_te)
pred_te = (p_te >= THRESH).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred_te, labels=[0, 1]).ravel()

RESULTS = dict(
    test_n=int(len(y_te)), test_defects=int(y_te.sum()), test_prevalence=float(y_te.mean()),
    threshold=THRESH, epochs=EPOCHS, **{f"hp_{k}": v for k, v in HP.items()},
    f1=float(f1_score(y_te, pred_te, zero_division=0)),
    precision=float(precision_score(y_te, pred_te, zero_division=0)),
    recall=float(recall_score(y_te, pred_te, zero_division=0)),
    roc_auc=float(roc_auc_score(y_te, p_te)),
    pr_auc=float(average_precision_score(y_te, p_te)),
    alarm_rate=float(pred_te.mean()),
    TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
)
for k, v in RESULTS.items():
    print(f"{k:16s} {v:.4f}" if isinstance(v, float) else f"{k:16s} {v}")
print(f"\nPR-AUC lift over no-skill baseline: {RESULTS['pr_auc'] / RESULTS['test_prevalence']:.2f}x")

trained 20 epochs on 1192 lots | final train loss 1.3879
test_n           313
test_defects     17
test_prevalence  0.0543
threshold        0.0033
epochs           20
hp_hidden        256
hp_latent        32
hp_lr            0.0010
f1               0.2326
precision        0.1923
recall           0.2941
roc_auc          0.7470
pr_auc           0.1713
alarm_rate       0.0831
TN               275
FP               21
FN               12
TP               5

PR-AUC lift over no-skill baseline: 3.15x


## 9 · Is the F1 real? Checks on the metric itself

A low F1 should be checked, not taken on trust. Each check below targets one way the number could be wrong.

In [10]:
# (a) recompute F1 by hand from the confusion matrix
f1_manual = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
assert abs(f1_manual - RESULTS["f1"]) < 1e-12
print(f"(a) F1 from confusion matrix = 2·{tp} / (2·{tp} + {fp} + {fn}) = {f1_manual:.4f}  ✔ matches sklearn")

# (b) label direction: 1 must mean 'defective' (label +1) and scores must not be inverted
assert y_te.sum() == (df['label'].to_numpy()[TEST['val']] == 1).sum()
print(f"(b) positives in test = {y_te.sum()} lots with label +1 ✔ | ROC-AUC with flipped scores = "
      f"{roc_auc_score(y_te, -p_te):.3f} (would be > 0.5 if the sign were wrong)")

# (c) positive control: the same code CAN produce a high F1 when the model is allowed to see the answers
ctrl_model, _, _ = fit_sae(TEST["val"], TEST["freeze"], HP, MAX_EPOCHS)
p_ctrl = predict(ctrl_model, z_te, m_te, c_te)
print(f"(c) positive control (trained AND scored on the test lots, invalid by design): "
      f"F1@0.5 = {f1_score(y_te, (p_ctrl >= 0.5).astype(int)):.3f}, PR-AUC = {average_precision_score(y_te, p_ctrl):.3f}"
      f"  -> the pipeline is able to score highly")

# (d) how good could ANY threshold have been? (oracle, picked on test labels, NOT a valid result)
pc, rc, _ = precision_recall_curve(y_te, p_te)
oracle_f1 = np.max(2 * pc * rc / np.clip(pc + rc, 1e-12, None))
print(f"(d) best F1 at any threshold on the test lots (oracle, invalid): {oracle_f1:.3f}")

# (e) uncertainty: 17 defects is very few -> bootstrap 95 % intervals
rng = np.random.default_rng(SEED); bs_f1, bs_ap = [], []
for _ in range(2000):
    i = rng.integers(0, len(y_te), len(y_te))
    if y_te[i].sum() == 0:
        continue
    bs_f1.append(f1_score(y_te[i], pred_te[i], zero_division=0)); bs_ap.append(average_precision_score(y_te[i], p_te[i]))
CI = dict(f1=np.percentile(bs_f1, [2.5, 97.5]).tolist(), pr_auc=np.percentile(bs_ap, [2.5, 97.5]).tolist())
RESULTS.update(f1_ci_low=CI["f1"][0], f1_ci_high=CI["f1"][1], pr_auc_ci_low=CI["pr_auc"][0], pr_auc_ci_high=CI["pr_auc"][1],
               oracle_f1=float(oracle_f1))
print(f"(e) bootstrap 95 % CI: F1 [{CI['f1'][0]:.3f}, {CI['f1'][1]:.3f}] | PR-AUC [{CI['pr_auc'][0]:.3f}, {CI['pr_auc'][1]:.3f}]")

# (f) seed sensitivity of the final model (primary result above uses the pre-declared SEED)
seed_rows = []
for s in (1, 2, 3, 4, 5):
    mdl, _, _ = fit_sae(TEST["train"], TEST["freeze"], HP, EPOCHS, seed=s)
    ps = predict(mdl, z_te, m_te, c_te)
    seed_rows.append(dict(seed=s, pr_auc=average_precision_score(y_te, ps), roc_auc=roc_auc_score(y_te, ps),
                          f1=f1_score(y_te, (ps >= THRESH).astype(int), zero_division=0)))
seed_df = pd.DataFrame(seed_rows)
print("(f) other seeds:\n" + seed_df.round(3).to_string(index=False))
RESULTS.update(seed_pr_auc_min=float(seed_df.pr_auc.min()), seed_pr_auc_max=float(seed_df.pr_auc.max()),
               seed_f1_min=float(seed_df.f1.min()), seed_f1_max=float(seed_df.f1.max()))

(a) F1 from confusion matrix = 2·5 / (2·5 + 21 + 12) = 0.2326  ✔ matches sklearn
(b) positives in test = 17 lots with label +1 ✔ | ROC-AUC with flipped scores = 0.253 (would be > 0.5 if the sign were wrong)


(c) positive control (trained AND scored on the test lots, invalid by design): F1@0.5 = 1.000, PR-AUC = 1.000  -> the pipeline is able to score highly
(d) best F1 at any threshold on the test lots (oracle, invalid): 0.258


(e) bootstrap 95 % CI: F1 [0.061, 0.391] | PR-AUC [0.084, 0.368]


(f) other seeds:
 seed  pr_auc  roc_auc    f1
    1   0.160    0.722 0.200
    2   0.176    0.764 0.200
    3   0.162    0.766 0.125
    4   0.127    0.704 0.182
    5   0.210    0.731 0.170


### Baseline under the identical protocol

A class-weighted logistic regression gets the same features, folds, embargo and threshold procedure.

In [11]:
def lr_fit(idx, freeze, C):
    z, m, c, y = build(idx, freeze)
    return LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(np.hstack([z, m, c]), y.astype(int))

def lr_predict(model, idx, freeze):
    z, m, c, _ = build(idx, freeze)
    return model.predict_proba(np.hstack([z, m, c]))[:, 1]

lr_scores = {}
for C in (1e-3, 1e-2, 1e-1, 1.0):
    oof = np.concatenate([lr_predict(lr_fit(f["train"], f["freeze"], C), f["val"], f["freeze"]) for f in FOLDS])
    lr_scores[C] = (average_precision_score(y_oof, oof), oof)
C_BEST = max(lr_scores, key=lambda C: lr_scores[C][0])
p_, r_, t_ = precision_recall_curve(y_oof, lr_scores[C_BEST][1])
f_ = 2 * p_[:-1] * r_[:-1] / np.clip(p_[:-1] + r_[:-1], 1e-12, None)
LR_THRESH = float(t_[np.nanargmax(f_)])
lr_final = lr_fit(TEST["train"], TEST["freeze"], C_BEST)
p_lr = lr_predict(lr_final, TEST["val"], TEST["freeze"])
pred_lr = (p_lr >= LR_THRESH).astype(int)
BASELINE = dict(C=C_BEST, oof_ap=lr_scores[C_BEST][0], threshold=LR_THRESH,
                f1=f1_score(y_te, pred_lr, zero_division=0), precision=precision_score(y_te, pred_lr, zero_division=0),
                recall=recall_score(y_te, pred_lr), roc_auc=roc_auc_score(y_te, p_lr), pr_auc=average_precision_score(y_te, p_lr))
print({k: round(float(v), 4) for k, v in BASELINE.items()})
RESULTS.update({f"baseline_{k}": float(v) for k, v in BASELINE.items()})

{'C': 0.1, 'oof_ap': 0.0637, 'threshold': 0.8589, 'f1': 0.1538, 'precision': 0.2222, 'recall': 0.1176, 'roc_auc': 0.724, 'pr_auc': 0.1413}


## 10 · Evidence: reproducing the original defects

Every number below is computed in this run.

In [12]:
# --- Defect 2: broadcast in the original reconstruction loss ---
xb = torch.randn(64, 1, 590)                         # original loader: unsqueeze(1) -> [B,1,F]
rec = torch.randn(64, 590, requires_grad=True)       # original model: flatten -> [B,F]
print(f"[bug 2] (reconstructed - batch_x).shape = {tuple((rec - xb).shape)}  <- should be (64, 590)")
g_bad,  = torch.autograd.grad(torch.mean((rec - xb) ** 2), rec)
g_good, = torch.autograd.grad(torch.mean((rec - xb.squeeze(1)) ** 2), rec)
g_mean, = torch.autograd.grad(torch.mean((rec - xb.mean(0)) ** 2), rec)
cos = lambda a, b: torch.nn.functional.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()
print(f"[bug 2] cosine(grad_broken, grad_correct) = {cos(g_bad, g_good):.3f}; "
      f"cosine(grad_broken, grad_toward_batch_mean) = {cos(g_bad, g_mean):.3f}")
print("[bug 2] -> the broken loss trains the decoder to output the batch average, not each wafer")

[bug 2] (reconstructed - batch_x).shape = (64, 64, 590)  <- should be (64, 590)
[bug 2] cosine(grad_broken, grad_correct) = 0.716; cosine(grad_broken, grad_toward_batch_mean) = 1.000
[bug 2] -> the broken loss trains the decoder to output the batch average, not each wafer


In [13]:
# --- Defect 3: PS-L sentinel / codes treated as measurements ---
raw0, raw1 = X_all[:, 0], X_all[:, 1]
print(f"[bug 3] sensor_0 mean/std with sentinel: {np.nanmean(raw0):8.1f} / {np.nanstd(raw0):6.1f}"
      f" | current-gen only: {np.nanmean(raw0[~is_legacy]):8.1f} / {np.nanstd(raw0[~is_legacy]):6.1f}")
print(f"[bug 3] sensor_1 mean/std with codes   : {np.nanmean(raw1):8.1f} / {np.nanstd(raw1):6.1f}"
      f" | current-gen only: {np.nanmean(raw1[~is_legacy]):8.1f} / {np.nanstd(raw1[~is_legacy]):6.1f}")
print(f"[bug 3] original scaling puts every PS-L sentinel at z = {(0 - np.nanmean(raw0)) / np.nanstd(raw0):.2f}, "
      f"and reads code 300 as three times code 100")

[bug 3] sensor_0 mean/std with sentinel:   2732.1 /  880.4 | current-gen only:   3014.0 /   74.3
[bug 3] sensor_1 mean/std with codes   :   2280.5 /  675.1 | current-gen only:   2495.7 /   79.8
[bug 3] original scaling puts every PS-L sentinel at z = -3.10, and reads code 300 as three times code 100


In [14]:
# --- Defect 1: leakage. Re-run the ORIGINAL preprocessing order and measure what it does ---
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

raw = pd.read_csv(DATA_PATH)
X_raw = raw[SENSOR_COLS].to_numpy(np.float64); y_raw = (raw["label"].to_numpy() == 1).astype(int)
Xs = StandardScaler().fit_transform(SimpleImputer(strategy="mean", keep_empty_features=True).fit_transform(X_raw))
Xb, yb_ = SMOTE(random_state=SEED).fit_resample(Xs, y_raw)
Xtr_o, Xte_o, ytr_o, yte_o = train_test_split(Xb, yb_, test_size=0.2, stratify=yb_, random_state=SEED)
n_synth = sum(not np.any(np.all(np.isclose(Xs, r), axis=1)) for r in Xte_o[yte_o == 1])
print(f"[bug 1] original test set: n={len(yte_o)} | defect rate {yte_o.mean():.1%} (natural is {y_raw.mean():.1%})")
print(f"[bug 1] {n_synth}/{(yte_o == 1).sum()} test 'defects' are SMOTE synthetics interpolated from training defects")
print(f"[bug 1] imputer/scaler were fit on all {len(X_raw)} rows, including future and test lots")
lr_leaky = LogisticRegression(max_iter=2000, C=0.1).fit(Xtr_o, ytr_o)
print(f"[bug 1] a plain logistic regression under the original protocol: F1 = {f1_score(yte_o, lr_leaky.predict(Xte_o)):.3f} "
      f"-> the F1 ≥ 0.90 gate measured the leak, not the model")

[bug 1] original test set: n=586 | defect rate 50.0% (natural is 6.6%)
[bug 1] 274/293 test 'defects' are SMOTE synthetics interpolated from training defects
[bug 1] imputer/scaler were fit on all 1567 rows, including future and test lots


[bug 1] a plain logistic regression under the original protocol: F1 = 0.932 -> the F1 ≥ 0.90 gate measured the leak, not the model


## 11 · Release gate, artifacts and a deployment check

In [15]:
GATE_F1 = 0.90
verdict = "PASSED ✅" if RESULTS["f1"] >= GATE_F1 else "FAILED ❌ — NO-GO"
print(f"Release gate (F1 ≥ {GATE_F1}) on the honest future test window: {RESULTS['f1']:.4f} → {verdict}")

# Artifacts use plain formats only (state_dict + .npz + .json), so `inference.py` can load them anywhere.
os.makedirs("artifacts", exist_ok=True)
torch.save(final_model.state_dict(), "artifacts/wafer_sae.pt")
med_f, scale_f, usable_f = stats_at(TEST["freeze"])
np.savez("artifacts/preprocess_state.npz", keep=KEEP, median=med_f, scale=scale_f, usable=usable_f)
with open("artifacts/config.json", "w") as fh:
    json.dump(dict(sensor_cols=SENSOR_COLS, legacy_codes=LEGACY_CODES, cat_names=CAT_NAMES, clip_z=CLIP_Z,
                   threshold=THRESH, d_cont=F_CONT, d_cat=CAT.shape[1], dropout=BASE_HP["dropout"],
                   hidden=HP["hidden"], latent=HP["latent"], stats_frozen_before=str(df.timestamp.iloc[TEST["freeze"]])),
              fh, indent=2)
with open("results.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2)
print("wrote artifacts/wafer_sae.pt, artifacts/preprocess_state.npz, artifacts/config.json, results.json")

# Deployment check: the standalone scorer must reproduce the notebook's test predictions exactly.
test_csv = "artifacts/_test_window.csv"
df.iloc[TEST["val"]].to_csv(test_csv, index=False)          # same time-sorted lots as above
out = subprocess.run([sys.executable, "inference.py", test_csv, "artifacts/_test_scores.csv"],
                     capture_output=True, text=True)
assert out.returncode == 0, out.stderr
scored = pd.read_csv("artifacts/_test_scores.csv")
assert np.allclose(scored["defect_probability"].to_numpy(), p_te, atol=1e-6)
assert (scored["flag"].to_numpy() == pred_te).all()
print(f"inference.py reproduces all {len(p_te)} test predictions ✔ (max abs diff "
      f"{np.abs(scored['defect_probability'].to_numpy() - p_te).max():.2e})")

Release gate (F1 ≥ 0.9) on the honest future test window: 0.2326 → FAILED ❌ — NO-GO
wrote artifacts/wafer_sae.pt, artifacts/preprocess_state.npz, artifacts/config.json, results.json


inference.py reproduces all 313 test predictions ✔ (max abs diff 2.98e-08)
